In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.customers (
    customer_id STRING NOT NULL,
    customer_name STRING NOT NULL,
    email STRING,
    registration_date DATE,
    customer_type STRING
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.products (
    product_id STRING NOT NULL,
    product_name STRING NOT NULL,
    category STRING,
    subcategory STRING,
    cost_price DOUBLE
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.orders (
    order_id STRING NOT NULL,
    customer_id STRING NOT NULL,
    order_date TIMESTAMP,
    status STRING,
    region_code STRING
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.order_items (
    item_id STRING NOT NULL,
    order_id STRING NOT NULL,
    product_id STRING NOT NULL,
    quantity INT NOT NULL,
    unit_price DOUBLE,
    discount_percent DOUBLE,
    is_return BOOLEAN
);

In [0]:
from pyspark.sql.functions import col, to_date

customers_path = "/Volumes/workspace/default/ecommerce_data/cleaned/customers_clean.csv"

customers_df = (
    spark.read
    .option("header", True)
    .csv(customers_path)
)

customers_df = customers_df.withColumn(
    "registration_date",
    to_date(col("registration_date"))
)

customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- customer_type: string (nullable = true)



In [0]:
customers_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.customers")

print("Customers loaded successfully")

Customers loaded successfully


In [0]:
%sql
SELECT COUNT(*) AS customer_count
FROM workspace.default.customers;

customer_count
500


In [0]:
from pyspark.sql.functions import col

products_path = "/Volumes/workspace/default/ecommerce_data/cleaned/products_clean.csv"

products_df = (
    spark.read
    .option("header", True)
    .csv(products_path)
)

products_df = products_df.withColumn(
    "cost_price",
    col("cost_price").cast("double")
)

products_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- cost_price: double (nullable = true)



In [0]:
products_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.products")

print("Products loaded successfully")

Products loaded successfully


In [0]:
%sql
SELECT COUNT(*) AS product_count
FROM workspace.default.products;

product_count
500


In [0]:
from pyspark.sql.functions import col, to_timestamp

orders_path = "/Volumes/workspace/default/ecommerce_data/cleaned/orders_clean.csv"

orders_df = (
    spark.read
    .option("header", True)
    .csv(orders_path)
)

orders_df = orders_df.withColumn(
    "order_date",
    to_timestamp(col("order_date"))
)

orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- region_code: string (nullable = true)



In [0]:
orders_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.orders")

print("Orders loaded successfully")

Orders loaded successfully


In [0]:
%sql
SELECT COUNT(*) AS order_count
FROM workspace.default.orders;

order_count
946


In [0]:
from pyspark.sql.functions import col

order_items_path = "/Volumes/workspace/default/ecommerce_data/cleaned/order_items_clean.csv"

order_items_df = (
    spark.read
    .option("header", True)
    .csv(order_items_path)
)

order_items_df = (
    order_items_df
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("unit_price", col("unit_price").cast("double"))
    .withColumn("discount_percent", col("discount_percent").cast("double"))
    .withColumn("is_return", col("is_return").cast("boolean"))
)

order_items_df.printSchema()

root
 |-- item_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- is_return: boolean (nullable = true)



In [0]:
order_items_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.order_items")

print("Order items loaded successfully")

Order items loaded successfully


In [0]:
%sql
SELECT COUNT(*) AS item_count
FROM workspace.default.order_items;

item_count
1880


In [0]:
%sql
SELECT 'customers' AS table_name, COUNT(*) AS row_count
FROM workspace.default.customers

UNION ALL

SELECT 'products', COUNT(*)
FROM workspace.default.products

UNION ALL

SELECT 'orders', COUNT(*)
FROM workspace.default.orders

UNION ALL

SELECT 'order_items', COUNT(*)
FROM workspace.default.order_items;

table_name,row_count
customers,500
products,500
orders,946
order_items,1880


In [0]:
%sql
SELECT COUNT(*) AS invalid_customer_ids
FROM workspace.default.orders o
LEFT JOIN workspace.default.customers c
    ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

invalid_customer_ids
0


In [0]:
%sql
SELECT COUNT(*) AS invalid_order_ids
FROM workspace.default.order_items oi
LEFT JOIN workspace.default.orders o
    ON oi.order_id = o.order_id
WHERE o.order_id IS NULL;

invalid_order_ids
0


In [0]:
%sql
SELECT COUNT(*) AS invalid_product_ids
FROM workspace.default.order_items oi
LEFT JOIN workspace.default.products p
    ON oi.product_id = p.product_id
WHERE p.product_id IS NULL;

invalid_product_ids
0
